# 03 - Temporal VAE + k-means (weekly scalograms)

Trains the Temporal VAE and saves its regimes. It does not analyze them: every metric is computed afterwards from the saved `regimes.csv` by `scripts/06_regime_metrics.py`, and the smoothed and scaled inputs by notebook 06.

Same as notebook 03, but each day's scalogram covers a week: day t and the 6 days before it (168 hours). The CNN encodes each day's scalogram, a one-direction LSTM reads the days in order, and the latent mean of day t comes from the day's own features concatenated with the LSTM state. Day t never sees days after t.

Loss = reconstruction + beta * KL + lambda * smoothness, where smoothness is the squared change of the latent mean between consecutive days relative to its spread (Nie et al., 2026).

Pipeline: 30 weekly scalograms (one per day) -> CNN per day -> causal LSTM -> latent mean per day + daily mean and std of the log real price -> k-means -> regime

Methodology notes:
1. Days are split chronologically (60% train, 20% validation, 20% test). The 2023-2024 El Nino falls in the test period.
2. The model is trained on sequences of 30 consecutive days. Each day's vector is computed from the scalograms of that day and the 29 days before it, the same context the model saw in training. Each of those scalograms covers 7 days, so the vector uses the price of that day and the 35 days before it.
3. The scalograms come from the log real price with the mean of the 7-day window removed, so they show the shape of the price inside the week, including the changes from one day to the next. The first 6 days of the series have no weekly scalogram, so there are 9703 days instead of 9709. The daily mean and standard deviation of the log real price, z-scored on training days, are added to the latent vector before clustering.
4. K-means is fitted on training days only and then assigns every day. Regimes are numbered by ascending mean real price, so the highest number is the most expensive regime.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

import itertools
import json
from pathlib import Path

import torch

import numpy as np
import altair as alt
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split, ScalogramSequenceDataset
from src.models.temporal_vae import TemporalVAE
from src.training import set_seed, train, active_dims
from src.clustering import with_level, level_columns
from src.plots import loss_curves, plot_reconstructions, regime_timeline
from src.runs import prepare_runs, save_all, cluster_runs, tsne_grid, regime_strips
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment configuration

Change only this cell between runs.
- `grid` holds the values of `lam`, `beta` and `seed`. The model is trained once for every combination and clustered for every k in `k_values`. The run folder of each combination ends with its values, e.g. `lam100_beta0.5_seed42`. The other hyperparameters are in `config` and are the same for every combination.
- `k_values` is always 3, 4 and 5.
- `study` names the folder `experiments/temporal_vae/kmeans/<study>/` and is saved in `config.json` and `registry.csv`.
- `tag` is an optional short label added to the run folder name before the grid values. Leave it empty to omit it.
- `load_from` is the `run_id` of an earlier run. When set, training is skipped, the grid is ignored and that run's model, history and latents are clustered again; its hyperparameters are used, so only `k_values`, `study` and `tag` come from this cell. An empty `tag` takes the tag of the loaded run.
- `active_threshold`: a latent dim is active when its KL on the training days at the last epoch is at least this value (nats).

In [ ]:
scalogram_set = "real_price_log_weekly"
k_values = [3, 4, 5]
study = "weekly_large"
tag = ""
load_from = None
active_threshold = 0.75

grid = {
   "lam": [100.0],
   "beta": [0.25, 0.5, 1.0],
   "seed": [42, 43, 44],
}

config = {
    "model": "temporal_vae",
    "study": study,
    "scalogram_set": scalogram_set,
    "k_values": k_values,
    "z_dim": 16,
    "filters": [128, 64, 32],
    "proj_dim": 128,
    "rnn_type": "lstm",
    "rnn_hidden": 64,
    "seq_len": 30,
    "stride": 14,
    "kl_warmup_epochs": 5,
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 16,
    "epochs": 30,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
}

## 1. Combinations

Every combination of the values in `grid`. Its run folders are created when its training starts. Section 4 shows the loss curves of every model and the reconstructions of the models with seed 42, and section 5 their regimes (or those of the loaded model).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if load_from and not tag:
    tag = json.loads((exp.EXPERIMENTS_DIR / load_from / "config.json").read_text()).get("tag", "")
combos = [{}] if load_from else [dict(zip(grid, values)) for values in itertools.product(*grid.values())]
print(f"{len(combos)} trainings x {len(k_values)} k = {len(combos) * len(k_values)} runs")
print(f"Device: {device}")

## 2. Data

In [ ]:
scalogram_dir = Path("../data/processed/scalograms") / config["scalogram_set"]
data_dir = Path("../data/processed")

images, metadata = load_scalograms(scalogram_dir)
context = load_context(data_dir, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), config["train_ratio"], config["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

img_c, img_h, img_w = images.shape[1:]
print(f"Scalograms ({config['scalogram_set']}): {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")
print(f"Daily level columns: {level_columns(metadata)}")
splits = exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx})
print(splits)

In [ ]:
train_set = ScalogramSequenceDataset(images, train_idx, config["seq_len"], config["stride"])
val_set = ScalogramSequenceDataset(images, val_idx, config["seq_len"], config["stride"])
train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)
print(f"Sequences of {config['seq_len']} days: train {len(train_set)}, val {len(val_set)}")

In [ ]:
def bce_floor(x, chunk=500):
    """Computed in blocks of `chunk` days: the weekly images in float64 do not fit in memory all at once."""
    floors = []
    for start in range(0, len(x), chunk):
        block = np.clip(x[start:start + chunk].reshape(-1, x[0].size).astype(np.float64), 1e-7, 1 - 1e-7)
        floors.append(-(block * np.log(block) + (1 - block) * np.log(1 - block)).sum(axis=1))
    return np.concatenate(floors)

for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    f = bce_floor(images[idx])
    print(f"{name}: floor {f.mean():.0f}, mean pixel {images[idx].mean():.3f}")

## 3. Training and clustering

For every combination: build the model, train it (skipped with `load_from`), extract the latent vectors and cluster once for each k. Each run folder gets:
- `config.json`, `history.json`, `regimes.csv` (one row per day, see `runs.regime_table`), `metrics.json` (silhouette, Davies-Bouldin and Calinski-Harabasz on the clustering features) and its row in `registry.csv`
- `tables/`: hyperparameters, regime profile (days, share, real price, reservoir, ONI and El Nino share of each regime) and clustering metrics
- `figures/`: loss curves, reconstructions of eight test days, t-SNE and regime timeline
- `artifacts/` (not in git): model weights and latent vectors

The printed line shows the KL of each latent dim at the last epoch (`kl_dim0` is the first dim of the vector, and so on) and how many dims are active (KL >= `active_threshold`). Each combination also runs a t-SNE of 5000 days, about one more minute.

In [ ]:
shown, trained_runs = {}, {}
for i, combo in enumerate(combos, 1):
    run_tag = "_".join(filter(None, [tag] + [f"{key}{value:g}" for key, value in combo.items()]))
    run_config, run_dirs, run_configs, trained = prepare_runs({**config, **combo, "tag": run_tag}, load_from, device)
    set_seed(run_config["seed"])
    trained_in = run_configs[k_values[0]]["trained_in"]
    show_model = bool(load_from) or run_config["seed"] == 42
    print(f"[{i}/{len(combos)}] {run_tag or load_from}: trained_in {trained_in}")

    model = TemporalVAE(run_config["z_dim"], img_c, img_h, img_w, run_config["filters"],
                        run_config["proj_dim"], run_config["rnn_hidden"], run_config["rnn_type"]).to(device)
    if trained:
        model.load_state_dict(trained["state_dict"])
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    for k, run_dir in run_dirs.items():
        run_configs[k]["n_params"] = n_params
        exp.save_config(run_dir, run_configs[k])

    if trained:
        history = trained["history"]
    else:
        history = train(model, train_loader, val_loader, device, epochs=run_config["epochs"], lr=run_config["lr"],
                        beta=run_config["beta"], kl_warmup_epochs=run_config["kl_warmup_epochs"],
                        extra_loss_kwargs={"lam": run_config["lam"]})
    save_all(run_dirs, exp.save_history, history)
    save_all(run_dirs, exp.save_model, model)
    loss_plot = loss_curves(history, components=("total", "recon", "kl", "smooth"))
    save_all(run_dirs, lambda d: loss_plot.save(exp.figure_path(d, "loss_curves"), scale_factor=2.0))
    trained_runs[trained_in] = {"loss_curves": loss_plot}

    model.eval()
    start = test_idx[len(test_idx) // 2]
    seq = torch.from_numpy(images[start:start + run_config["seq_len"]]).unsqueeze(0).to(device)
    with torch.no_grad():
        recon = model(seq)[0][0].cpu().numpy()
    days8 = np.arange(run_config["seq_len"] - 8, run_config["seq_len"])
    fig = plot_reconstructions(images[start + days8], recon[days8],
                               [str(d.date()) for d in context["date"].iloc[start + days8]])
    save_all(run_dirs, lambda d: fig.savefig(exp.figure_path(d, "reconstructions"), dpi=150, bbox_inches="tight"))
    if show_model:
        trained_runs[trained_in]["reconstructions"] = fig
    plt.close(fig)

    latents = trained["latents"] if trained else model.extract_latents(images, device, seq_len=run_config["seq_len"])
    save_all(run_dirs, exp.save_latents, latents)
    n_active, kl_dims = active_dims(history, active_threshold)
    print(f"    KL per dim {kl_dims}, active dims {n_active} of {run_config['z_dim']}")

    features = with_level(latents, metadata, train_idx)
    rng = np.random.default_rng(run_config["seed"])
    plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
    coords = TSNE(n_components=2, perplexity=30, random_state=run_config["seed"]).fit_transform(features[plot_idx])
    results = cluster_runs(run_dirs, run_configs, features, context, train_idx, split_dates, coords, plot_idx)
    if show_model:
        shown[run_tag or load_from] = {"results": results, "coords": coords, "plot_idx": plot_idx}

## 4. Loss curves and reconstructions

First the loss curves of every trained model (total, reconstruction, KL and smoothness on training and validation days), each under the `run_id` of the run where it was trained. Then the reconstructions of eight test days of the models with seed 42 (or of the loaded model): original scalograms on top, reconstructed below. Both figures are also saved in `figures/` of every run of the model.

In [ ]:
for run_id, figures in trained_runs.items():
    print(run_id)
    display(figures["loss_curves"])

for run_id, figures in trained_runs.items():
    if "reconstructions" in figures:
        print(run_id)
        display(figures["reconstructions"])

## 5. Regimes of the models with seed 42

For each shown model: the t-SNE of the clustering features colored by the regimes of k = 3, 4 and 5, the regime timeline of each k (real price colored by regime, regime strip and reservoir level, with El Nino periods shaded and the split boundaries dashed), and the regime strips of the three k together.

In [ ]:
for name, s in shown.items():
    print(name)
    display(tsne_grid(s["coords"], s["plot_idx"], s["results"]))
    for k, ctx in s["results"].items():
        display(regime_timeline(ctx, split_dates, title=f"{name}: detected regimes over time (k={k})"))
    display(regime_strips(s["results"]))